# 🤖 Gemma 4 Tech Interviewer — Colab API Server

This notebook loads your fine-tuned Gemma 4 model from Hugging Face and serves it as a FastAPI server exposed via `ngrok`.

## Steps:
1. Run **Cell 1** to install dependencies.
2. Run **Cell 2** to check GPU.
3. Add your `HF_TOKEN`, `NGROK_TOKEN`, and `NGROK_DOMAIN` secrets in the 🔑 key icon on the left sidebar.
   - Get `NGROK_TOKEN` from [dashboard.ngrok.com/authtokens](https://dashboard.ngrok.com/authtokens)
   - Get a free static `NGROK_DOMAIN` from [dashboard.ngrok.com/domains](https://dashboard.ngrok.com/domains)
4. Run **Cell 3** to load the model.
5. Run **Cell 4** to start the API server and get your public URL.
6. Paste the URL into your backend `.env` as `GEMMA_API_URL=<url>`.

In [ ]:
# CELL 1: Install dependencies
!pip install -q transformers peft bitsandbytes accelerate fastapi uvicorn pyngrok huggingface_hub
print('Dependencies installed!')

In [ ]:
# CELL 2: Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found! Go to Runtime -> Change runtime type -> A100 GPU')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 3: Load fine-tuned Gemma 4 model
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

BASE_MODEL_ID = 'google/gemma-4-e2b-it'
ADAPTER_ID = 'Mohamud24/gemma-4-tech-interviewer'

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID, token=HF_TOKEN)

print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': 0},
    torch_dtype=compute_dtype,
    token=HF_TOKEN
)

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base_model, ADAPTER_ID, token=HF_TOKEN)
model.eval()

print('Model ready!')

In [ ]:
# CELL 4: Start FastAPI server + ngrok tunnel
import json
import re
import uvicorn
from threading import Thread
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok, conf
from google.colab import userdata

# Setup ngrok
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
NGROK_DOMAIN = userdata.get('NGROK_DOMAIN')  # claim free static domain at dashboard.ngrok.com/domains
conf.get_default().auth_token = NGROK_TOKEN

app = FastAPI(title='Gemma 4 Tech Interviewer API')

class InterviewRequest(BaseModel):
    endpoint: str
    payload: dict

def generate_response(prompt: str, max_tokens: int = 800) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def build_prompt(task: str, payload: dict) -> str:
    candidate = payload.get('candidate_name', 'Candidate')
    lang = payload.get('language', 'en')
    specialization = payload.get('specialization', payload.get('jobRole', payload.get('domain', 'technology')))
    difficulty = payload.get('difficulty', 'mid')
    question = payload.get('question', '')
    answer = payload.get('answer', '')

    messages = [
        {'role': 'user', 'content': (
            f'task: {task}\n'
            f'candidate_name: {candidate}\n'
            f'language: {lang}\n'
            f'specialization: {specialization}\n'
            f'difficulty: {difficulty}\n'
            + (f'question: {question}\n' if question else '')
            + (f'answer: {answer}\n' if answer else '')
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

TASK_MAP = {
    '/ask_technical_question': 'ask_technical_question',
    '/score_candidate_answer': 'score_candidate_answer',
    '/open_mock_interview_session': 'open_mock_interview_session',
    '/close_mock_interview_session': 'close_mock_interview_session',
    '/open_hiring_interview_session': 'open_hiring_interview_session',
    '/close_hiring_interview_session': 'close_hiring_interview_session',
}

@app.get('/health')
def health():
    return {'status': 'online', 'model': 'Mohamud24/gemma-4-tech-interviewer', 'provider': 'colab'}

@app.post('/runsync')
async def runsync(req: InterviewRequest):
    task_key = req.endpoint
    if task_key not in TASK_MAP:
        raise HTTPException(status_code=404, detail=f'Unknown endpoint: {task_key}')
    task = TASK_MAP[task_key]
    prompt = build_prompt(task, req.payload)
    response = generate_response(prompt)
    return {'output': {'response': response, 'task': task}}

# Start uvicorn in background thread to avoid Colab asyncio conflicts
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Connect ngrok tunnel with static domain
tunnel = ngrok.connect(8000, 'http', domain=NGROK_DOMAIN)
public_url = tunnel.public_url

print(f'\n==========================================')
print(f'YOUR COLAB GEMMA URL IS READY!')
print(f'URL: {public_url}')
print(f'==========================================')
print(f'Paste this into your backend .env file:')
print(f'GEMMA_API_URL={public_url}')
print(f'==========================================')